In [1]:
# Please install OpenAI SDK first: `pip3 install openai`
from openai import OpenAI
API_KEY = "sk-99f0c3fc32a4413cbf01a4ab90e2ee6a"

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Hello"},
    ],
    stream=False
)

print(response.choices[0].message.content)

Hello! How can I assist you today? 😊


In [8]:
type(response.choices[0])

openai.types.chat.chat_completion.Choice

In [ ]:
import openai.types.chat.chat_completion as Message
Message = Message.Choice
isinstance(response.choices[0],Message)

True

: 

In [7]:
response.choices[0].__dict__

{'finish_reason': 'stop',
 'index': 0,
 'logprobs': None,
 'message': ChatCompletionMessage(content='Hello! How can I assist you today? 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)}

In [6]:
response.choices[0].message.__dict__

{'content': 'Hello! How can I assist you today? 😊',
 'refusal': None,
 'role': 'assistant',
 'annotations': None,
 'audio': None,
 'function_call': None,
 'tool_calls': None}

In [4]:
dir(response.choices[0].message)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__',
 '__pydantic_serializer__

In [11]:
for block in response.choices[0].message.content:
    print(block.type)

AttributeError: 'str' object has no attribute 'type'

In [ ]:
from openai import OpenAI
API_KEY = "sk-99f0c3fc32a4413cbf01a4ab90e2ee6a"
import openai.types.chat.chat_completion_message as Message

class Deepseek:
    def __init__(self,api_key:str,model:str):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://api.deepseek.com"
        )
        self.model = model

    def add_user_message(self,messages:list,message):
        user_message = {
            "role":"user",
            "content":message.content if isinstance(message,Message.ChatCompletionMessage)
            else message
        }
        messages.append(user_message)

    def add_assistant_message(self,messages:list,message):
        assistant_message = {
            "role":"assistant",
            "content":message.content if isinstance(message,Message.ChatCompletionMessage)
            else message
        }
        messages.append(assistant_message)
    
    def chat(
        self,
        messages,
        system = None,
        temperature = 1.0,
        tools = None    
    ):
        params = {
            "model":self.model,
            "messages":messages,
            "temperature":temperature
        }

        if system:
            params['system'] = system

        if tools:
            params['tools'] = tools

        response = self.client.chat.completions.create(**params)
        return response.choices[0].message.content

In [17]:
deepseek = Deepseek(api_key=API_KEY,model="deepseek-chat")
messages = []
deepseek.add_user_message(messages,"Hello, how are you?")
response = deepseek.chat(messages=messages)
print(response)

Hello! I'm doing well, thank you for asking! 😊 I'm here and ready to help you with anything you might need—whether it's answering questions, brainstorming ideas, or just having a friendly chat. How are you doing today?


In [18]:
deepseek.add_assistant_message(messages,response)
deepseek.add_user_message(messages,"tell me how to loss weight")
response = deepseek.chat(messages)


In [21]:
messages

[{'role': 'user', 'content': 'Hello, how are you?'},
 {'role': 'assistant',
  'content': "Hello! I'm doing well, thank you for asking! 😊 I'm here and ready to help you with anything you might need—whether it's answering questions, brainstorming ideas, or just having a friendly chat. How are you doing today?"},
 {'role': 'user', 'content': 'tell me how to loss weight'},
 {'role': 'assistant',
  'content': "Of course! Losing weight in a healthy, sustainable way involves a combination of diet, exercise, and lifestyle changes. It's important to focus on long-term habits rather than quick fixes.\n\nHere’s a practical, step-by-step guide:\n\n### 1. **Understand the Basics**\n   - **Calorie Deficit:** Weight loss happens when you consume fewer calories than you burn. You can achieve this by eating less, moving more, or both.\n   - **Safe Rate:** Aim to lose **0.5–1 kg (1–2 lbs)** per week. Faster weight loss can lead to muscle loss, nutritional deficiencies, and is harder to maintain.\n\n### 

In [7]:
isinstance(response.choices[0].message,Message.ChatCompletionMessage)

True

## 工具调用

In [22]:
from openai import OpenAI

def send_messages(messages):
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        tools=tools
    )
    return response.choices[0].message

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.deepseek.com",
)

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather of a location, the user should supply a location first.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    }
                },
                "required": ["location"]
            },
        }
    },
]

messages = [{"role": "user", "content": "How's the weather in Hangzhou, Zhejiang?"}]
message = send_messages(messages)
print(f"User>\t {messages[0]['content']}")

tool = message.tool_calls[0]
messages.append(message)

messages.append({"role": "tool", "tool_call_id": tool.id, "content": "24℃"})
message = send_messages(messages)
print(f"Model>\t {message.content}")

User>	 How's the weather in Hangzhou, Zhejiang?
Model>	 The current weather in Hangzhou, Zhejiang is 24°C. This is a pleasant, mild temperature that's comfortable for most outdoor activities.


In [23]:
print(tool)

ChatCompletionMessageFunctionToolCall(id='call_00_SzotVYtRRmeH8BIgft53R1gS', function=Function(arguments='{"location": "Hangzhou, Zhejiang"}', name='get_weather'), type='function', index=0)
